# 03 — COCO → YOLO + 3-way split (train/val/test, leakage-free & stratified)

Converts `aug/annotations_coco.json` to **YOLO format** and splits **70/15/15** into
train/val/test.

### Design decisions
- **nc/names derived dynamically from the COCO** (so a new category in the export is
  handled automatically; order is `0=face, 1=license-plate, 2=car, ...`).
- **Leakage-free:** the split is done at the **source-photo level** (the stem before
  `_r..._######_v#`). All tiles/variants of one original photo land in **exactly one**
  split. Deterministic (seed 42).
- **Stratified (multi-label, greedy):** each source photo is assigned to the split that
  is most under-quota for the classes it contains, so `face` AND `license-plate` end up
  in **all three** splits (especially val & test).
- **Images as symlinks** (no duplication, saves disk).
- Final asserts: all 3 splits non-empty, and `face` + `license-plate` present in val AND test.

## 1. Configuration & load COCO

In [ ]:
import os, json, re, random, glob
from collections import Counter, defaultdict

ROOT = "/home/jovyan/shared/s0598584"
AUG_DIR = os.path.join(ROOT, "aug")
DATASET = os.path.join(ROOT, "dataset")

SPLIT_FRACS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42
IMGSZ = 1280

for sub in ["images/train","images/val","images/test","labels/train","labels/val","labels/test"]:
    os.makedirs(os.path.join(DATASET, sub), exist_ok=True)

with open(os.path.join(AUG_DIR, "annotations_coco.json")) as f:
    coco = json.load(f)
print("aug images:", len(coco["images"]), "| anns:", len(coco["annotations"]))
print("COCO categories:", [(c["id"], c["name"]) for c in sorted(coco['categories'],key=lambda c:c['id'])])

## 2. Class mapping (multi-class, derived from COCO)

In [ ]:
cats = sorted(coco["categories"], key=lambda c: c["id"])
coco_to_yolo = {c["id"]: i for i, c in enumerate(cats)}
names = {i: c["name"] for i, c in enumerate(cats)}
NC = len(names)
names_inv = {v: k for k, v in names.items()}
FACE_Y = names_inv["face"]; LP_Y = names_inv["license-plate"]
print("nc =", NC)
print("names =", names)
print("coco_to_yolo =", coco_to_yolo)
print("face->yolo", FACE_Y, "| license-plate->yolo", LP_Y)

## 3. Determine source photos + per-photo class inventory

In [ ]:
def source_key(fn):
    m = re.match(r"(.+?)_r[-0-9p\.]+_\d+_v\d+\.png$", fn)
    return m.group(1) if m else fn

anns_by_img = defaultdict(list)
for a in coco["annotations"]:
    anns_by_img[a["image_id"]].append(a)

source_of = {im["id"]: source_key(im["file_name"]) for im in coco["images"]}

src_imgcount = Counter()
src_classcount = defaultdict(Counter)   # src -> Counter{yolo_cls: n_anns}
for im in coco["images"]:
    s = source_of[im["id"]]
    src_imgcount[s] += 1
    for a in anns_by_img.get(im["id"], []):
        src_classcount[s][coco_to_yolo[a["category_id"]]] += 1

sources = sorted(src_imgcount.keys())
print("source photos total:", len(sources))
print("tiles total        :", sum(src_imgcount.values()))
n_src_face = sum(1 for s in sources if src_classcount[s].get(FACE_Y,0)>0)
n_src_lp   = sum(1 for s in sources if src_classcount[s].get(LP_Y,0)>0)
print(f"sources with face: {n_src_face} | with license-plate: {n_src_lp}")

## 4. Greedy multi-label stratified, leakage-free split

We sort source photos by the rarity of their classes (rarest first, so they are
distributed fairly first) and assign each photo to the split most under-quota for its
classes relative to the target fraction. Tie-break: the split with the largest remaining
tile deficit. Deterministic (seed 42).

In [ ]:
rng = random.Random(SEED)

global_cls = Counter()
for s in sources:
    for c, n in src_classcount[s].items():
        global_cls[c] += n

def rarity(s):
    cs = src_classcount[s]
    if not cs:
        return (10**9, 0)
    rarest = min(global_cls[c] for c in cs)
    return (rarest, -sum(cs.values()))

order = sorted(sources, key=lambda s: (rarity(s), rng.random()))

splits = ["train", "val", "test"]
target_tiles = {sp: SPLIT_FRACS[sp] * sum(src_imgcount.values()) for sp in splits}
cur_tiles = {sp: 0 for sp in splits}
target_cls = {sp: {c: SPLIT_FRACS[sp]*global_cls[c] for c in global_cls} for sp in splits}
cur_cls = {sp: Counter() for sp in splits}
assign = {}

def deficit_score(s, sp):
    score = 0.0
    for c, n in src_classcount[s].items():
        tgt = target_cls[sp][c]
        if tgt <= 0:
            continue
        deficit = (tgt - cur_cls[sp][c]) / tgt
        score += deficit * n
    tdef = (target_tiles[sp] - cur_tiles[sp]) / max(1.0, target_tiles[sp])
    return score + 0.25 * tdef * src_imgcount[s]

for s in order:
    best_sp = max(splits, key=lambda sp: deficit_score(s, sp))
    assign[s] = best_sp
    cur_tiles[best_sp] += src_imgcount[s]
    for c, n in src_classcount[s].items():
        cur_cls[best_sp][c] += n

src_per_split = Counter(assign.values())
print("source photos per split:", dict(src_per_split))
print("tiles per split (planned):", cur_tiles)
for sp in splits:
    print(f"  {sp}: face={cur_cls[sp][FACE_Y]} license-plate={cur_cls[sp][LP_Y]}")

for sp in ["val", "test"]:
    assert cur_cls[sp][FACE_Y] > 0, f"face missing in {sp}!"
    assert cur_cls[sp][LP_Y] > 0, f"license-plate missing in {sp}!"
print("OK: face & license-plate present in val AND test.")

def split_of(img_id):
    return assign[source_of[img_id]]

## 5. Write YOLO labels + image symlinks

In [ ]:
counts = {sp: 0 for sp in splits}
label_dist = {sp: Counter() for sp in splits}

for im in coco["images"]:
    sp = split_of(im["id"]); fn = im["file_name"]
    W = im["width"]; H = im["height"]
    src_img = os.path.join(AUG_DIR, "images", fn)
    dst_img = os.path.join(DATASET, "images", sp, fn)
    if os.path.lexists(dst_img):
        os.remove(dst_img)
    os.symlink(src_img, dst_img)

    lines = []
    for a in anns_by_img.get(im["id"], []):
        ycls = coco_to_yolo[a["category_id"]]
        x, y, w, h = a["bbox"]
        xc = min(max((x + w/2)/W, 0.0), 1.0)
        yc = min(max((y + h/2)/H, 0.0), 1.0)
        nw = min(max(w/W, 0.0), 1.0)
        nh = min(max(h/H, 0.0), 1.0)
        if nw <= 0 or nh <= 0:
            continue
        lines.append(f"{ycls} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
        label_dist[sp][ycls] += 1
    lbl_path = os.path.join(DATASET, "labels", sp, os.path.splitext(fn)[0] + ".txt")
    with open(lbl_path, "w") as f:
        f.write("\n".join(lines))
    counts[sp] += 1

print("images per split:", counts)

## 6. dataset.yaml (3-way split, full multi-class)

In [ ]:
import yaml
data = {
    "path": DATASET,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": NC,
    "names": names,
}
yaml_path = os.path.join(DATASET, "dataset.yaml")
with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
print(open(yaml_path).read())

## 7. Verification: counts, per-class distribution, leakage, asserts

> This produces the **full multi-class** dataset. The 2-class (`face`, `license-plate`)
> dataset used to train the shipped model is derived from this in a follow-up step
> (`dataset_face_lp`), which keeps all face/plate tiles plus a controlled fraction of
> car/person-only tiles as hard-negative backgrounds.

In [ ]:
tr_imgs = glob.glob(os.path.join(DATASET, "images/train/*.png"))
va_imgs = glob.glob(os.path.join(DATASET, "images/val/*.png"))
te_imgs = glob.glob(os.path.join(DATASET, "images/test/*.png"))
print(f"train: {len(tr_imgs)} | val: {len(va_imgs)} | test: {len(te_imgs)}")
for sp, lst in [("train",tr_imgs),("val",va_imgs),("test",te_imgs)]:
    lbls = glob.glob(os.path.join(DATASET, f"labels/{sp}/*.txt"))
    assert len(lst) == len(lbls) and len(lst) > 0, f"{sp}: image/label mismatch or empty"

print("\nPer-class distribution per split (annotations):")
print(f"  {'class':16s} {'train':>8s} {'val':>8s} {'test':>8s}")
for cid in sorted(names):
    print(f"  {names[cid]:16s} {label_dist['train'][cid]:8d} {label_dist['val'][cid]:8d} {label_dist['test'][cid]:8d}")

tr_src = {source_key(os.path.basename(p)) for p in tr_imgs}
va_src = {source_key(os.path.basename(p)) for p in va_imgs}
te_src = {source_key(os.path.basename(p)) for p in te_imgs}
print("\nLeakage train&val:", len(tr_src & va_src), "| train&test:", len(tr_src & te_src), "| val&test:", len(va_src & te_src))
assert tr_src.isdisjoint(va_src) and tr_src.isdisjoint(te_src) and va_src.isdisjoint(te_src)

assert len(tr_imgs)>0 and len(va_imgs)>0 and len(te_imgs)>0
for sp in ["val","test"]:
    assert label_dist[sp][FACE_Y] > 0, f"face missing in {sp}"
    assert label_dist[sp][LP_Y] > 0, f"license-plate missing in {sp}"
print("\nAll asserts green: 3 splits filled, leakage-free, face & lp in val+test.")

## ✅ Phase 3 done

YOLO dataset under `dataset/` with a leakage-free, stratified 70/15/15 split +
`dataset.yaml`. Next: build the 2-class `dataset_face_lp` and run `04_batch_finder.ipynb`.